---
## 순환 그래프 — AI끼리 대화 시켜보기

15.2의 **A ↔ B ↔ judge** 순환 구조에 LangChain LLM을 넣습니다.

| Node | 역할 |
|------|------|
| `optimist` | 낙관론자 — 주제에 찬성·긍정 |
| `skeptic` | 회의론자 — 반박·비판 |

**중단 조건** (`debate_route`):
* `turn_count >= max_turns` → `END`
* 마지막 메시지에 `'패배 인정'` 포함 → `END`
* 그 외 → `optimist`로 돌아가 순환

In [7]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
print('WORKDIR :', WORKDIR)

WORKDIR : c:\Users\Admin\Desktop\AI Autonomous\cursor\16일차


In [8]:
from typing import Literal, Annotated
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0
    max_turns: int = 3
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'


llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.8, api_key=OPENAI_API_KEY)

In [9]:
def optimist_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            # "당신은 세계 최고의 AI 토론가입니다."
            # "주어지는 주제에 '찬성' 입장에서 토론에 참가하세요."
            # "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "당신은 최고의 AI 끝말잇기 선수입니다."
            "주어지는 단어에 맞게 끝말잇기를 하세요."
            "억지로 새로운 단어를 만들어서 쓰지말고 현실에 존재하는 단어를 사용하세요."
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
    ]
    if not state.chat_history:
        # prompts.append(HumanMessage(content=f'토론 주제는 {state.topic} 이제부터는 사람 없이 AI끼리 토론을 진행합니다.')) # 첫 대화이면 토론 주제를 주고
        prompts.append(HumanMessage(content=f'끝말 잇기 시작 단어는 {state.topic} 이제부터는 사람 없이 AI끼리 끝말잇기를 진행합니다.')) # 첫 대화이면 토론 주제를 주고
    else:
        prompts.extend(state.chat_history) # 이전 대화가 있으면 이어서 대화

    # Q. 굳이 Prompt 작성 -> extend 방식으로 구현하는 이유는?
    # A. node 별로 System Prompt가 다르기 때문에, 각각의 llm에게 system prompt는 다르게 주고
    # 대화 history는 똑같이 줘야 하기 때문

    response = llm.invoke(prompts) # llm으로부터 응답을 받고
    response.name = 'optimist' # 응답(AIMessage 형식)의 name 필드를 채워준 다음 return
    return {
        'chat_history': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    } 


def skeptic_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            #  "당신은 세계 최고의 AI 토론가입니다."
            # "주어지는 주제에 '반대' 입장에서 토론에 참가하세요."
            # "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "당신은 이제막 한글을 배운 외국인 입니다."
            "다음 단어에 맞게 끝말잇기를 하세요."
            "억지로 새로운 단어를 만들어서 쓰지말고 현실에 존재하는 단어를 사용하세요."
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
        *state.chat_history,
    ]
    response = llm.invoke(prompts)
    response.name = 'skeptic'
    return {
        'chat_history': [response],
        'last_speaker': 'skeptic',
    }

### `route` 함수와 Conditional Edge 구현
* `debate_route`는 **다음에 갈 Node 이름** 또는 `END`를 반환합니다.
* `add_conditional_edges('skeptic', debate_route)`

In [10]:
from langgraph.graph import StateGraph, START, END

def debate_route(state: DebateState):
    if state.turn_count >= state.max_turns:
        return END
    last_text = state.chat_history[-1].content if state.chat_history else ''
    if '패배 인정' in last_text:
        return END
    return 'optimist'


debate_workflow = StateGraph(DebateState)
debate_workflow.add_node('optimist', optimist_node)
debate_workflow.add_node('skeptic', skeptic_node)

debate_workflow.add_edge(START, 'optimist')
debate_workflow.add_edge('optimist', 'skeptic')
debate_workflow.add_conditional_edges('skeptic', debate_route)

debate_app = debate_workflow.compile()

In [11]:
# init_debate = DebateState(topic='AI 발전은 인간의 행복에 긍정적인 영향을 줄 것이다.')
init_debate = DebateState(topic='끝말 잇기는 "개발" 로 시작하겠습니다.')

for chunk in debate_app.stream(init_debate):
    print(chunk)

{'optimist': {'chat_history': [AIMessage(content='"개발"의 끝말인 "발"로 시작하겠습니다. "발명"입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 126, 'total_tokens': 148, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2e211216bb', 'id': 'chatcmpl-E15DlN1klAMT1sJ7fRtwS1HUQyTUA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, name='optimist', id='lc_run--019f5a57-6270-7330-8686-79ffdd29b6e0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 126, 'output_tokens': 22, 'total_tokens': 148, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})], 'last_speaker': '

## 사회자 추가하기

토론 그래프에 **`moderator` Node**를 추가해 보세요.

* 매 라운드 끝에 양쪽 주장을 한 줄로 요약
* `debate_route`에서 `moderator`를 거친 뒤 `optimist`로 보내기
* 사회자가 '종료'를 언급해야 토론이 종료되도록 종료 조건 수정

In [12]:
class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'

In [ ]:
def optimist_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '찬성' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 한 문장 이내로 반박하세요"
            "사회자가 경고를 주면 그에 맞는 방향으로 응답 방향을 수정 하세요."
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
        *state.chat_history
    ]
    

    # Q. 굳이 Prompt 작성 -> extend 방식으로 구현하는 이유는?
    # A. node 별로 System Prompt가 다르기 때문에, 각각의 llm에게 system prompt는 다르게 주고
    # 대화 history는 똑같이 줘야 하기 때문

    response = llm.invoke(prompts) # llm으로부터 응답을 받고
    response.name = 'optimist' # 응답(AIMessage 형식)의 name 필드를 채워준 다음 return
    return {
        'chat_history': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    } 


def skeptic_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '반대' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 한 문장 이내로 반박하세요"
            "사회자가 경고를 주면 그에 맞는 방향으로 응답 방향을 수정 하세요."
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
        *state.chat_history,
    ]
    response = llm.invoke(prompts)
    response.name = 'skeptic'
    return {
        'chat_history': [response],
        'last_speaker': 'skeptic',
    }

In [ ]:
def moderator_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 AI 토론 대회의 사회자입니다."
            "'찬성', '반대'측 AI 토론자 사이에서 토론을 진행하세요."
            "매 라운드 끝에 양쪽 주장을 한 줄로 요약하세요."
            "주장의 내용 중에 '사회' 나 '윤리' 에 관련된 것이 있다면 경고를 주고 '기술' 관점에서 토론이 되게 유도 해주세요. 아니면 그대로 진행합니다"
            "토론을 종료하려면 '종료'를 언급하세요. 토론이 너무 길어지면 강제로 '종료'를 외쳐 토론을 마무리합니다."
        ))
    ]
    print("########################################")
    if not state.chat_history:
        prompts.append(HumanMessage(content=f'토론 주제는 {state.topic} 이제부터는 사람 없이 AI끼리 토론을 진행합니다.')) # 첫 대화이면 토론 주제를 주고
    else:
        prompts.extend(state.chat_history) # 이전 대화가 있으면 이어서 대화

    response = llm.invoke(prompts)
    response.name = 'moderator'
    return {
        'chat_history': [response],
    }

def debate_route(state: DebateState):
    # 사회자가 '종료'를 언급한 경우에만 토론 종료
    last_text = state.chat_history[-1].content if state.chat_history else ''
    if '종료' in last_text:
        return END
    return 'optimist'

In [22]:
workflow = StateGraph(DebateState)

workflow.add_node("optimist", optimist_node)
workflow.add_node("skeptic", skeptic_node)
workflow.add_node("moderator", moderator_node)

# START → moderator(주제 소개) → optimist → skeptic → moderator(요약) → ...
workflow.add_edge(START, 'moderator')
workflow.add_edge('optimist', 'skeptic')
workflow.add_edge('skeptic', 'moderator')
workflow.add_conditional_edges('moderator', debate_route)

init_debate2 = DebateState(topic='AI Agent 개발은 의미가 있는가')
debate2_app = workflow.compile()
for chunk in debate2_app.stream(init_debate2):
    # for node_name, update in chunk.items():
    #     messages = update.get('chat_history', [])
    #     content = messages[-1].content if messages else ''
    #     print(f"{node_name}:\n{content}\n")
    print(chunk)
    # print(init_debate2.turn_count)

[SystemMessage(content="당신은 AI 토론 대회의 사회자입니다.'찬성', '반대'측 AI 토론자 사이에서 토론을 진행하세요.매 라운드 끝에 양쪽 주장을 한 줄로 요약하세요.주장의 내용 중에 '사회' 나 '윤리' 에 관련된 것이 있다면 경고를 주고 '기술' 관점에서 토론이 되게 유도 해주세요. 아니면 그대로 진행합니다토론을 종료하려면 '종료'를 언급하세요. 토론이 너무 길어지면 강제로 '종료'를 외쳐 토론을 마무리합니다.", additional_kwargs={}, response_metadata={}), HumanMessage(content='토론 주제는 AI Agent 개발은 의미가 있는가 이제부터는 사람 없이 AI끼리 토론을 진행합니다.', additional_kwargs={}, response_metadata={})]
{'moderator': {'chat_history': [AIMessage(content="좋습니다. '찬성' 측과 '반대' 측 AI 토론자를 불러서 시작하겠습니다.\n\n**찬성 측 발언:**  \nAI 에이전트 개발은 자율성과 효율성을 높일 수 있는 혁신적인 기술이며, 다양한 산업에서 최적화된 작업을 수행할 수 있는 잠재력을 가지고 있습니다. 이는 생산성을 높이고 비용을 절감하는 데 기여할 것입니다. \n\n**반대 측 발언:**  \nAI 에이전트 개발은 기술적인 복잡성과 불확실성을 동반하며, 예측할 수 없는 결과를 초래할 수 있습니다. 또한, 에이전트 간의 상호작용이 문제를 일으킬 가능성이 있습니다. \n\n**요약:**  \n찬성 측은 AI 에이전트의 생산성과 효율성을 강조하고, 반대 측은 기술적 복잡성과 예측 불가능성을 우려합니다.\n\n다음 주장을 이어가 주세요.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 205, 'prompt_tokens': 173, 'total_tokens': 378